In [1]:
import torch
from torch import nn
from pathlib import Path
from model_def import get_model
from test_mnist import download, load_images, load_labels  # reuse these

In [5]:
def load_mnist_train(root="./data"):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    image_file = "train-images-idx3-ubyte.gz"
    label_file = "train-labels-idx1-ubyte.gz"
    image_path = root / image_file
    label_path = root / label_file

    mnist_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"
    download(mnist_url + image_file, image_path)
    download(mnist_url + label_file, label_path)

    images = load_images(image_path)
    labels = load_labels(label_path)

    images = images[:, None, :, :]
    images = images.repeat(3, axis=1)

    return images, labels

In [10]:
def train():
    images, labels = load_mnist_train()
    X = torch.tensor(images, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)

    model = get_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    batch_size = 64
    n = X.shape[0]

    for epoch in range(10):
        perm = torch.randperm(n)          # shuffle indices each epoch
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = X[idx], y[idx]

            optimizer.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch}, avg loss {total_loss / (n / batch_size):.4f}")

    torch.save(model.state_dict(), "trained.pt")
    print("saved trained.pt")
train()

epoch 0, avg loss 0.3972
epoch 1, avg loss 0.2921
epoch 2, avg loss 0.2775
epoch 3, avg loss 0.2705
epoch 4, avg loss 0.2650
epoch 5, avg loss 0.2616
epoch 6, avg loss 0.2585
epoch 7, avg loss 0.2555
epoch 8, avg loss 0.2545
epoch 9, avg loss 0.2520
saved trained.pt


epoch 0, avg loss 0.3971
epoch 1, avg loss 0.2914
epoch 2, avg loss 0.2783
epoch 3, avg loss 0.2699
epoch 4, avg loss 0.2655
saved trained.pt
